# CylinderFit 2026 — Tutorial

This notebook walks through the main capabilities of `cylfit`:

1. Basic fitting from a NumPy array
2. Fitting with outliers (RANSAC robustness)
3. Loading from PLY / PCD files
4. Open3D adapter
5. Known-radius and constrained-axis fitting
6. Multi-cylinder detection
7. Fit quality metrics and uncertainty estimation
8. Parallel RANSAC
9. Visualization
10. Exporting (JSON, mesh)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cylfit import (
    fit_cylinder,
    fit_cylinder_known_radius,
    fit_cylinder_constrained_axis,
    fit_cylinder_with_normals,
    fit_cylinder_auto,
    detect_cylinders,
    generate_noisy_cylinder,
    measure_fit,
    evaluate_fit,
    estimate_uncertainty,
    load_points,
    from_open3d,
)

rng = np.random.default_rng(42)

## 1. Basic fitting

`generate_noisy_cylinder` creates synthetic ground-truth data. In practice you would load a real point cloud.

In [ ]:
syn = generate_noisy_cylinder(
    radius=2.5,
    height=8.0,
    noise=0.015,
    outlier_fraction=0.0,
    random_state=42,
)
print(f"True radius: {syn.radius:.3f}  True axis: {syn.axis_direction}")

model = fit_cylinder(syn.points, threshold=0.06, ransac_trials=64, random_state=42)

print(f"Fitted radius: {model.radius:.4f}")
print(f"Fitted axis:   {model.axis_direction}")
print(f"RMSE: {model.rmse:.5f}")
print(f"Inliers: {model.inlier_mask.sum()} / {len(model.inlier_mask)}")
print(f"Converged: {model.converged} in {model.iterations} iterations")

## 2. Robustness to outliers

30 % outlier contamination — RANSAC still recovers the cylinder.

In [ ]:
syn_dirty = generate_noisy_cylinder(
    radius=2.5, noise=0.02, outlier_fraction=0.30, random_state=7
)

model_dirty = fit_cylinder(
    syn_dirty.points, threshold=0.09, ransac_trials=128, random_state=7
)

metrics = evaluate_fit(model_dirty, syn_dirty)
print(f"Radius error:   {metrics.radius_error:.4f}")
print(f"Axis angle:     {metrics.axis_angle_deg:.3f}°")
print(f"Inlier recall:  {metrics.inlier_recall:.3f}")
print(f"Inlier precis.: {metrics.inlier_precision:.3f}")

## 3. Loading point cloud files

`load_points` auto-detects PLY (ASCII/binary), PCD (ASCII/binary), LAS/LAZ, and XYZ.

In [ ]:
import tempfile, os

# Write a temporary PLY and load it back
pts = syn.points
with tempfile.NamedTemporaryFile(suffix='.ply', mode='w', delete=False) as f:
    f.write('ply\nformat ascii 1.0\n')
    f.write(f'element vertex {len(pts)}\n')
    f.write('property float x\nproperty float y\nproperty float z\n')
    f.write('end_header\n')
    for row in pts:
        f.write(f'{row[0]} {row[1]} {row[2]}\n')
    ply_path = f.name

loaded = load_points(ply_path)
os.unlink(ply_path)

print(f"Loaded {loaded.shape[0]} points from PLY")
model_from_ply = fit_cylinder(loaded, threshold=0.06, ransac_trials=48, random_state=42)
print(f"Radius from PLY data: {model_from_ply.radius:.4f}")

## 4. Open3D adapter

`from_open3d` converts `open3d.geometry.PointCloud` → NumPy array without copying data.

In [ ]:
# Using a mock since open3d may not be installed in every environment
class MockO3DCloud:
    """Drop-in mock for open3d.geometry.PointCloud"""
    def __init__(self, pts): self.points = pts

cloud = MockO3DCloud(syn.points)
arr = from_open3d(cloud)
print(f"from_open3d: {arr.shape}, dtype={arr.dtype}")

# When open3d is installed:
# import open3d as o3d
# cloud = o3d.io.read_point_cloud('scan.ply')
# arr = from_open3d(cloud)
# model = fit_cylinder(arr)

## 5. Constrained and known-radius fitting

In [ ]:
# Known radius: only axis position and direction are optimized
model_kr = fit_cylinder_known_radius(
    syn.points, radius=2.5, threshold=0.06, ransac_trials=32, random_state=42
)
print(f"Known-radius fit: radius={model_kr.radius:.6f} (pinned to 2.5)")

# Axis constrained to ±8° from the true axis
model_ca = fit_cylinder_constrained_axis(
    syn.points,
    axis=syn.axis_direction,
    max_axis_angle_deg=8.0,
    threshold=0.06,
    ransac_trials=32,
    random_state=42,
)
angle = np.degrees(np.arccos(np.clip(abs(np.dot(model_ca.axis_direction, syn.axis_direction)), 0, 1)))
print(f"Constrained axis deviation: {angle:.3f}°  (limit: 8°)")

## 6. Multi-cylinder detection

In [ ]:
cyl_a = generate_noisy_cylinder(
    radius=0.8, height=4.0,
    axis_point=np.array([-3.0, 0.0, 0.0]),
    noise=0.01, n_points=1000, random_state=10,
)
cyl_b = generate_noisy_cylinder(
    radius=1.2, height=5.0,
    axis_point=np.array([3.0, 0.0, 0.0]),
    noise=0.01, n_points=1000, random_state=11,
)
mixed = np.vstack([cyl_a.points, cyl_b.points])

detections = detect_cylinders(
    mixed,
    max_cylinders=2,
    threshold=0.05,
    min_inliers=400,
    cluster_eps=0.5,
    ransac_trials=64,
    random_state=42,
)

print(f"Detected {len(detections)} cylinders:")
for i, det in enumerate(detections):
    print(f"  [{i}] radius={det.model.radius:.3f}  height={det.model.height:.3f}  "
          f"inliers={det.model.inlier_mask.sum()}")

## 7. Fit quality metrics and uncertainty estimation

In [ ]:
# FitMeasures: coverage, percentile residuals, quality score
measures = measure_fit(syn.points, model)
print(f"Angular coverage:    {measures.angular_coverage_deg:.1f}°")
print(f"Axial coverage:      {measures.axial_coverage:.3f}")
print(f"P90 residual:        {measures.residual_p90:.5f}")
print(f"Quality score:       {measures.quality_score:.1f} / 100")

# Bootstrap uncertainty
uncertainty = estimate_uncertainty(
    syn.points, model, samples=30, sample_fraction=0.7, random_state=42
)
print(f"\nRadius: {uncertainty.radius_mean:.4f} ± {uncertainty.radius_std:.4f}")
print(f"Radius 95% CI: [{uncertainty.radius_ci95[0]:.4f}, {uncertainty.radius_ci95[1]:.4f}]")
print(f"Axis angle std: {uncertainty.axis_angle_std_deg:.4f}°")

## 8. Parallel RANSAC

`n_jobs=-1` uses all CPU cores. Trials are split across threads; NumPy SVD releases the GIL.

In [ ]:
import time

large_syn = generate_noisy_cylinder(
    n_points=5000, radius=2.5, noise=0.02, outlier_fraction=0.2, random_state=0
)

t0 = time.perf_counter()
m_serial = fit_cylinder(large_syn.points, ransac_trials=128, n_jobs=1, random_state=0)
t_serial = time.perf_counter() - t0

t0 = time.perf_counter()
m_parallel = fit_cylinder(large_syn.points, ransac_trials=128, n_jobs=-1, random_state=0)
t_parallel = time.perf_counter() - t0

print(f"Serial:   {t_serial:.3f}s  radius={m_serial.radius:.4f}")
print(f"Parallel: {t_parallel:.3f}s  radius={m_parallel.radius:.4f}")
print(f"Speedup:  {t_serial / t_parallel:.2f}×")

## 9. Visualization

Requires `matplotlib` (`pip install cylfit[visualize]`).

In [ ]:
try:
    fig = model.plot(syn.points)
    plt.show()
except ImportError:
    print("Install matplotlib: pip install cylfit[visualize]")

## 10. Exporting results

In [ ]:
# JSON summary
print(model.to_json(indent=2))

In [ ]:
# Triangle mesh (e.g. for PyVista / Open3D)
vertices, faces = model.to_mesh(n_theta=64, n_height=16)
print(f"Mesh: {len(vertices)} vertices, {len(faces)} triangles")

# Start point / end point for pipe-network analysis
print(f"Start: {model.start_point}")
print(f"End:   {model.end_point}")
print(f"Height: {model.height:.4f}")